# exp_20260513_025_lgb_year_stint_flags_min

Base is `code_020.txt` / `exp_20260511_020_lgb_progress_window_te_min`.

This notebook preserves the 020 implementation and adds only explicit Year/Stint/Tyre-wear features.

No weather features, no pseudo label, no original prior, no `Normalized_TyreLife` or proxy.


In [1]:
# ============================================================
# PS S6E5 - exp_20260513_025_lgb_year_stint_flags_min
#
# Base:
#   020: exp_20260511_020_lgb_progress_window_te_min
#
# Change from 020:
#   Add discussion-derived Year/Stint/Tyre-wear explicit features.
#
# Keep from 020:
#   - shared_003-style full FE
#   - original rows appended to every outer-fold train side
#   - StratifiedKFold by target x Year
#   - seed ensemble [3407, 42]
#   - LightGBM params from 020
#   - progress-window count/frequency and TE from 020
#
# Do NOT add:
#   - weather features
#   - Race_Year_RaceProgressBin
#   - 72 bins
#   - original prior
#   - pseudo label
#   - Normalized_TyreLife or proxy
#   - Stint_x_Normalized / Norm_x_Hardness
#   - final stint length / endpoint / future proxy
# ============================================================

In [2]:
import os
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, TargetEncoder

import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260513_025_lgb_year_stint_flags_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    TEST_PATH = COMP_BASE / "test.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    ORIG_PATH = Path(
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv"
    )

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"
    DANGER_COL = "Normalized_TyreLife"

    SEED = 3407
    SEEDS_BASE = [3407, 42]
    N_SPLITS = 5

    USE_ORIGINAL_ROWS = True
    USE_TARGET_ENCODING = True

    # Same as 020.
    LGB_N_ESTIMATORS = 9000
    LGB_LEARNING_RATE = 0.022
    LGB_NUM_LEAVES = 31
    LGB_MIN_CHILD_SAMPLES = 120
    LGB_SUBSAMPLE = 0.88
    LGB_SUBSAMPLE_FREQ = 1
    LGB_COLSAMPLE_BYTREE = 0.82
    LGB_REG_ALPHA = 0.25
    LGB_REG_LAMBDA = 10.0
    LGB_MAX_BIN = 255
    LGB_EARLY_STOPPING = 450

    # Kaggle環境でGPU LightGBMが失敗する場合は自動でCPU fallback
    LGB_DEVICE = "gpu"

    # 020 progress-window features
    USE_PROGRESS_WINDOW_TE = True
    PROGRESS_BINS = 20

    PROGRESS_WINDOW_CAT_COLS = [
        "RaceProgressBin_20",
        "Race_RaceProgressBin_20",
        "Race_Compound_RaceProgressBin_20",
    ]

    PROGRESS_WINDOW_TE_COLS = [
        "Race_RaceProgressBin_20",
        "Race_Compound_RaceProgressBin_20",
    ]

    # 025 explicit Year/Stint features.
    USE_YEAR_STINT_FLAGS_025 = True
    ADD_YEAR_STINT_CROSSES_025 = True
    ADD_YEAR_STINT_TE_025 = False  # intentionally false; avoid amplifying Year artifact by TE


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in out.columns:
        dtype = out[col].dtype

        if pd.api.types.is_integer_dtype(dtype):
            c_min = out[col].min()
            c_max = out[col].max()

            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                out[col] = out[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                out[col] = out[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                out[col] = out[col].astype(np.int32)

        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")

    return out


def safe_divide(a, b, eps=1e-6):
    return a / (b + eps)


def print_section(title: str) -> None:
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load data
# ============================================================

print_section("Load Data")

train_raw = pd.read_csv(CFG.TRAIN_PATH)
test_raw = pd.read_csv(CFG.TEST_PATH)
sub = pd.read_csv(CFG.SUB_PATH)
orig_raw = pd.read_csv(CFG.ORIG_PATH)

print("train_raw:", train_raw.shape)
print("test_raw :", test_raw.shape)
print("orig_raw :", orig_raw.shape)
print("target mean competition:", train_raw[CFG.TARGET].mean())
print("target mean original   :", orig_raw[CFG.TARGET].mean())

assert CFG.ID_COL in train_raw.columns
assert CFG.ID_COL in test_raw.columns
assert CFG.TARGET in train_raw.columns
assert CFG.TARGET in orig_raw.columns
assert CFG.DANGER_COL in orig_raw.columns

if CFG.DANGER_COL in orig_raw.columns:
    orig_raw = orig_raw.drop(columns=[CFG.DANGER_COL])

train_raw = reduce_mem_usage(train_raw)
test_raw = reduce_mem_usage(test_raw)
orig_raw = reduce_mem_usage(orig_raw)

test_ids = test_raw[CFG.ID_COL].copy()
train_ids = train_raw[CFG.ID_COL].copy()


Load Data
train_raw: (439140, 16)
test_raw : (188165, 15)
orig_raw : (101371, 16)
target mean competition: 0.19898210137996994
target mean original   : 0.25479673673930414


In [6]:
# ============================================================
# 020: Progress-window features
# ============================================================

def add_progress_window_base_features(df: pd.DataFrame, bins: int = 20) -> pd.DataFrame:
    """
    Add RaceProgress window features.

    This function does not use target.
    It is safe to apply to train/test/original before CV.

    Important:
      - Do not add Year here.
      - Do not add Race_Year_RaceProgressBin.
      - Keep bins coarse enough to avoid overfitting.
    """
    out = df.copy()

    rp = pd.to_numeric(out["RaceProgress"], errors="coerce").clip(0, 1)

    # 0..bins-1
    bin_id = np.floor(rp * bins).astype("float")
    bin_id = bin_id.clip(0, bins - 1).fillna(-1).astype(int)

    out[f"RaceProgressBin_{bins}"] = bin_id.astype(str)

    out[f"Race_RaceProgressBin_{bins}"] = (
        out["Race"].astype(str) + "_" + out[f"RaceProgressBin_{bins}"].astype(str)
    )

    out[f"Race_Compound_RaceProgressBin_{bins}"] = (
        out["Race"].astype(str)
        + "_"
        + out["Compound"].astype(str)
        + "_"
        + out[f"RaceProgressBin_{bins}"].astype(str)
    )

    return out

In [7]:
# ============================================================
# 025: Year/Stint flags
# ============================================================

def add_year_stint_flags_025(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add explicit Year/Stint/Tyre-wear features.

    This function does not use target.
    It intentionally does NOT reconstruct Normalized_TyreLife or any endpoint proxy.
    """
    out = df.copy()

    if "Year" in out.columns:
        out["Is_2022_025"] = (out["Year"] == 2022).astype(np.int8)
        out["Is_2023_025"] = (out["Year"] == 2023).astype(np.int8)
        out["Is_2024_025"] = (out["Year"] == 2024).astype(np.int8)
        out["Is_2025_025"] = (out["Year"] == 2025).astype(np.int8)

    if "Stint" in out.columns:
        out["Is_Stint1_025"] = (out["Stint"] == 1).astype(np.int8)
        out["Is_Stint2_025"] = (out["Stint"] == 2).astype(np.int8)
        out["Is_Stint3_025"] = (out["Stint"] == 3).astype(np.int8)
        out["Is_Stint4Plus_025"] = (out["Stint"] >= 4).astype(np.int8)
        out["Is_Second_Stint_025"] = (out["Stint"] == 2).astype(np.int8)
        out["Is_Third_StintPlus_025"] = (out["Stint"] >= 3).astype(np.int8)

    if "TyreLife" in out.columns:
        tyre = pd.to_numeric(out["TyreLife"], errors="coerce")
        tyre_clip = tyre.clip(lower=0)
        out["TyreLife_sq_025"] = (tyre ** 2).astype(np.float32)
        out["TyreLife_sqrt_025"] = np.sqrt(tyre_clip).astype(np.float32)
        out["TyreLife_log1p_025"] = np.log1p(tyre_clip).astype(np.float32)

    if "LapTime_Delta" in out.columns:
        out["LapTime_Delta_abs_025"] = pd.to_numeric(out["LapTime_Delta"], errors="coerce").abs().astype(np.float32)

    if "Cumulative_Degradation" in out.columns:
        out["Cumulative_Degradation_abs_025"] = (
            pd.to_numeric(out["Cumulative_Degradation"], errors="coerce").abs().astype(np.float32)
        )

    if {"Stint", "RaceProgress"}.issubset(out.columns):
        out["Stint_x_RaceProgress_025"] = (
            pd.to_numeric(out["Stint"], errors="coerce") *
            pd.to_numeric(out["RaceProgress"], errors="coerce")
        ).astype(np.float32)

    if {"Stint", "TyreLife"}.issubset(out.columns):
        out["Stint_x_TyreLife_sq_025"] = (
            pd.to_numeric(out["Stint"], errors="coerce") *
            (pd.to_numeric(out["TyreLife"], errors="coerce") ** 2)
        ).astype(np.float32)

    if {"TyreLife", "RaceProgress"}.issubset(out.columns):
        out["TyreLife_sq_x_RaceProgress_025"] = (
            (pd.to_numeric(out["TyreLife"], errors="coerce") ** 2) *
            pd.to_numeric(out["RaceProgress"], errors="coerce")
        ).astype(np.float32)

    return out


YEAR_STINT_FLAG_FEATURES_025 = [
    "Is_2022_025",
    "Is_2023_025",
    "Is_2024_025",
    "Is_2025_025",
    "Is_Stint1_025",
    "Is_Stint2_025",
    "Is_Stint3_025",
    "Is_Stint4Plus_025",
    "Is_Second_Stint_025",
    "Is_Third_StintPlus_025",
    "TyreLife_sq_025",
    "TyreLife_sqrt_025",
    "TyreLife_log1p_025",
    "LapTime_Delta_abs_025",
    "Cumulative_Degradation_abs_025",
    "Stint_x_RaceProgress_025",
    "Stint_x_TyreLife_sq_025",
    "TyreLife_sq_x_RaceProgress_025",
]

YEAR_STINT_CROSS_FEATURES_025 = [
    "Year_Stint_025",
    "Year_Compound_025",
]

In [8]:
# ============================================================
# shared_003-style FE
# ============================================================

BASE_CAT_COLS = ["Driver", "Compound", "Race"]

NUMERIC_BASE_COLS = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]


def add_base_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6

    for col in BASE_CAT_COLS:
        if col in out.columns:
            out[col] = out[col].astype("string").fillna("__MISSING__").astype(str)

    # shared_003 baseline FE style
    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedTotalLaps"] = safe_divide(
            out["LapNumber"],
            out["RaceProgress"].clip(lower=eps),
            eps,
        ).replace([np.inf, -np.inf], np.nan).clip(1, 120)

        out["LapsRemaining"] = (
            out["EstimatedTotalLaps"] - out["LapNumber"]
        ).clip(lower=0)

    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreAgeRatio"] = safe_divide(
            out["TyreLife"],
            out["LapNumber"].clip(lower=1),
            eps,
        )

    if {"Cumulative_Degradation", "TyreLife"}.issubset(out.columns):
        out["DegPerTyreLap"] = safe_divide(
            out["Cumulative_Degradation"],
            out["TyreLife"].clip(lower=1),
            eps,
        )

    if {"Position", "RaceProgress"}.issubset(out.columns):
        out["PositionPressure"] = out["Position"] * out["RaceProgress"]

    # Additional compact full-FE features from shared_003 description
    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["LapProgress_x_LapNumber"] = out["LapNumber"] * out["RaceProgress"]

    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["LapPerTyreLife"] = safe_divide(out["LapNumber"], out["TyreLife"] + 1, eps)
        out["LapMinusTyreLife"] = out["LapNumber"] - out["TyreLife"]
        out["TyreLifeMinusLap"] = out["TyreLife"] - out["LapNumber"]

    if {"TyreLife", "EstimatedTotalLaps"}.issubset(out.columns):
        out["TyreAgeVsRace"] = safe_divide(
            out["TyreLife"],
            out["EstimatedTotalLaps"].clip(lower=1),
            eps,
        )

    if {"TyreLife", "RaceProgress"}.issubset(out.columns):
        out["PitWindowPressure"] = out["TyreLife"] * out["RaceProgress"]
        out["TyreLife_x_RaceProgress"] = out["TyreLife"] * out["RaceProgress"]

    if {"Cumulative_Degradation", "LapNumber"}.issubset(out.columns):
        out["DegPerRaceLap"] = safe_divide(
            out["Cumulative_Degradation"],
            out["LapNumber"].clip(lower=1),
            eps,
        )

    if {"LapTime_Delta", "TyreLife"}.issubset(out.columns):
        out["DeltaPerTyreLap"] = safe_divide(
            out["LapTime_Delta"],
            out["TyreLife"].clip(lower=1),
            eps,
        )

    if "LapTime_Delta" in out.columns:
        out["DeltaAbs"] = out["LapTime_Delta"].abs()
        out["LapTimeDeltaPositive"] = (out["LapTime_Delta"] > 0).astype(np.int8)
        out["LapTimeDeltaNegative"] = (out["LapTime_Delta"] < 0).astype(np.int8)

    if "Cumulative_Degradation" in out.columns:
        out["Abs_Cumulative_Degradation"] = out["Cumulative_Degradation"].abs()
        out["Positive_Degradation"] = (out["Cumulative_Degradation"] > 0).astype(np.int8)

    if "Position_Change" in out.columns:
        out["Abs_Position_Change"] = out["Position_Change"].abs()
        out["Gained_Position"] = (out["Position_Change"] > 0).astype(np.int8)
        out["Lost_Position"] = (out["Position_Change"] < 0).astype(np.int8)

    if {"Stint", "TyreLife"}.issubset(out.columns):
        out["StintPressure"] = out["Stint"] * out["TyreLife"]
        out["TyreLife_x_Stint"] = out["TyreLife"] * out["Stint"]

    if {"Stint", "LapNumber"}.issubset(out.columns):
        out["Stint_x_LapNumber"] = out["Stint"] * out["LapNumber"]
        out["Is_First_Stint"] = (out["Stint"] == 1).astype(np.int8)
        out["Is_Late_Stint"] = (out["Stint"] >= 3).astype(np.int8)

    if "RaceProgress" in out.columns:
        out["Early_Race"] = (out["RaceProgress"] <= 0.25).astype(np.int8)
        out["Mid_Race"] = (
            (out["RaceProgress"] > 0.25) &
            (out["RaceProgress"] <= 0.65)
        ).astype(np.int8)
        out["Late_Race"] = (out["RaceProgress"] > 0.65).astype(np.int8)

        out["RaceProgressBin"] = pd.cut(
            out["RaceProgress"],
            bins=[-np.inf, 0.20, 0.40, 0.60, 0.80, np.inf],
            labels=["P1", "P2", "P3", "P4", "P5"],
        ).astype(str)

    if "TyreLife" in out.columns:
        out["TyreLifeBin"] = pd.cut(
            out["TyreLife"],
            bins=[-np.inf, 3, 7, 12, 20, 30, np.inf],
            labels=["T_000_003", "T_004_007", "T_008_012", "T_013_020", "T_021_030", "T_031_plus"],
        ).astype(str)

    if "Year" in out.columns:
        out["Year_str"] = out["Year"].astype(str)
        out["is_2023"] = (out["Year"] == 2023).astype(np.int8)
        out["Year_minus_2022"] = (out["Year"] - 2022).astype(np.int8)

    # 025差分: explicit Year/Stint/Tyre-wear flags.
    if CFG.USE_YEAR_STINT_FLAGS_025:
        out = add_year_stint_flags_025(out)

    return out


def add_interaction_categories(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    def make_cross(name, cols):
        if set(cols).issubset(out.columns):
            s = out[cols[0]].astype(str)
            for c in cols[1:]:
                s = s + "_" + out[c].astype(str)
            out[name] = s

    make_cross("Driver_Compound", ["Driver", "Compound"])
    make_cross("Race_Compound", ["Race", "Compound"])
    make_cross("Race_Year", ["Race", "Year"])
    make_cross("Driver_Race", ["Driver", "Race"])
    make_cross("Driver_Year", ["Driver", "Year"])
    make_cross("Stint_Compound", ["Stint", "Compound"])
    make_cross("Compound_TyreLifeBin", ["Compound", "TyreLifeBin"])
    make_cross("Compound_RaceProgressBin", ["Compound", "RaceProgressBin"])

    # 025差分: minimal Year/Stint categorical crosses.
    # Do NOT add Race_Year_Stint or Driver_Year_Stint at first pass.
    if CFG.ADD_YEAR_STINT_CROSSES_025:
        make_cross("Year_Stint_025", ["Year", "Stint"])
        make_cross("Year_Compound_025", ["Year", "Compound"])

    return out


def add_frequency_features(all_df: pd.DataFrame) -> pd.DataFrame:
    out = all_df.copy()

    freq_cols = [
        "Driver",
        "Race",
        "Compound",
        "Driver_Race",
        "Race_Year",
        "Year_str",
    ]

    # 025差分: frequency only, no TE, for minimal Year/Stint crosses.
    if CFG.ADD_YEAR_STINT_CROSSES_025:
        freq_cols += [
            "Year_Stint_025",
            "Year_Compound_025",
        ]

    freq_cols = [c for c in freq_cols if c in out.columns]

    total = len(out)

    for col in freq_cols:
        counts = out[col].astype(str).value_counts(dropna=False)
        out[f"{col}_freq"] = (
            out[col].astype(str).map(counts).fillna(0) / total
        ).astype(np.float32)

    return out


def add_lag_features(all_df: pd.DataFrame) -> pd.DataFrame:
    out = all_df.copy()

    required = {"Driver", "Race", "Stint", "LapNumber"}
    if not required.issubset(out.columns):
        return out

    out["_orig_order_for_lag"] = np.arange(len(out))
    out = out.sort_values(["Driver", "Race", "Stint", "LapNumber", "_orig_order_for_lag"])

    group_cols = ["Driver", "Race", "Stint"]

    if "LapTime_Delta" in out.columns:
        out["Delta_lag1"] = (
            out.groupby(group_cols, sort=False)["LapTime_Delta"]
            .shift(1)
        )

        out["Delta_roll3_mean"] = (
            out.groupby(group_cols, sort=False)["LapTime_Delta"]
            .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
        )

    if "Cumulative_Degradation" in out.columns:
        out["Deg_diff"] = (
            out.groupby(group_cols, sort=False)["Cumulative_Degradation"]
            .diff()
        )

    if "TyreLife" in out.columns:
        out["TyreLife_growth"] = (
            out.groupby(group_cols, sort=False)["TyreLife"]
            .diff()
        )

    if "LapTime (s)" in out.columns:
        out["LapTime_lag1"] = (
            out.groupby(group_cols, sort=False)["LapTime (s)"]
            .shift(1)
        )

        out["LapTime_diff"] = (
            out.groupby(group_cols, sort=False)["LapTime (s)"]
            .diff()
        )

    out = out.sort_values("_orig_order_for_lag").drop(columns=["_orig_order_for_lag"])

    return out


def fill_missing_and_types(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.replace([np.inf, -np.inf], np.nan)

    for col in out.columns:
        if col in [CFG.ID_COL, CFG.TARGET, "_dataset"]:
            continue

        if out[col].dtype == "object" or str(out[col].dtype).startswith("category") or str(out[col].dtype).startswith("string"):
            out[col] = out[col].astype("string").fillna("__MISSING__").astype(str)
        else:
            med = out[col].median()
            out[col] = out[col].fillna(med)
            if out[col].dtype == "float64":
                out[col] = out[col].astype(np.float32)

    return reduce_mem_usage(out)


print_section("Feature Engineering")

train_comp = train_raw.copy()
test_comp = test_raw.copy()
orig_comp = orig_raw.copy()

train_comp["_dataset"] = "train"
test_comp["_dataset"] = "test"
orig_comp["_dataset"] = "orig"

# original has no id in the reference dataset in many versions.
# Add synthetic negative ids only for alignment.
if CFG.ID_COL not in orig_comp.columns:
    orig_comp[CFG.ID_COL] = -np.arange(1, len(orig_comp) + 1)

# Add missing target to test for concat.
test_comp[CFG.TARGET] = np.nan

all_df = pd.concat(
    [train_comp, orig_comp, test_comp],
    axis=0,
    ignore_index=True,
    sort=False,
)

print("all_df initial:", all_df.shape)

all_df = add_base_features(all_df)
all_df = add_interaction_categories(all_df)
all_df = add_frequency_features(all_df)
all_df = add_lag_features(all_df)
all_df = fill_missing_and_types(all_df)

print("all_df fe:", all_df.shape)

train_fe = all_df[all_df["_dataset"] == "train"].reset_index(drop=True).copy()
orig_fe = all_df[all_df["_dataset"] == "orig"].reset_index(drop=True).copy()
test_fe = all_df[all_df["_dataset"] == "test"].reset_index(drop=True).copy()


Feature Engineering
all_df initial: (728676, 17)
all_df fe: (728676, 94)


In [9]:
# ============================================================
# 020差分: RaceProgress window base features
# ============================================================

train_fe = add_progress_window_base_features(train_fe, bins=CFG.PROGRESS_BINS)
orig_fe = add_progress_window_base_features(orig_fe, bins=CFG.PROGRESS_BINS)
test_fe = add_progress_window_base_features(test_fe, bins=CFG.PROGRESS_BINS)

print("020 progress window base columns:")
for c in CFG.PROGRESS_WINDOW_CAT_COLS:
    print(
        c,
        "train:", c in train_fe.columns,
        "orig:", c in orig_fe.columns,
        "test:", c in test_fe.columns,
    )

print("025 Year/Stint features:")
for c in YEAR_STINT_FLAG_FEATURES_025 + YEAR_STINT_CROSS_FEATURES_025:
    print(
        c,
        "train:", c in train_fe.columns,
        "orig:", c in orig_fe.columns,
        "test:", c in test_fe.columns,
    )

# Guardrails: do not add Year interaction for progress windows
assert "Race_Year_RaceProgressBin_20" not in train_fe.columns
assert "TE_Race_Year_RaceProgressBin_20" not in train_fe.columns
assert CFG.DANGER_COL not in train_fe.columns

# Restore target dtypes
train_fe[CFG.TARGET] = train_raw[CFG.TARGET].astype(int).values
orig_fe[CFG.TARGET] = orig_raw[CFG.TARGET].astype(int).values

print("train_fe:", train_fe.shape)
print("orig_fe :", orig_fe.shape)
print("test_fe :", test_fe.shape)

020 progress window base columns:
RaceProgressBin_20 train: True orig: True test: True
Race_RaceProgressBin_20 train: True orig: True test: True
Race_Compound_RaceProgressBin_20 train: True orig: True test: True
025 Year/Stint features:
Is_2022_025 train: True orig: True test: True
Is_2023_025 train: True orig: True test: True
Is_2024_025 train: True orig: True test: True
Is_2025_025 train: True orig: True test: True
Is_Stint1_025 train: True orig: True test: True
Is_Stint2_025 train: True orig: True test: True
Is_Stint3_025 train: True orig: True test: True
Is_Stint4Plus_025 train: True orig: True test: True
Is_Second_Stint_025 train: True orig: True test: True
Is_Third_StintPlus_025 train: True orig: True test: True
TyreLife_sq_025 train: True orig: True test: True
TyreLife_sqrt_025 train: True orig: True test: True
TyreLife_log1p_025 train: True orig: True test: True
LapTime_Delta_abs_025 train: True orig: True test: True
Cumulative_Degradation_abs_025 train: True orig: True test: T

In [10]:
# ============================================================
# Feature list and encoding
# ============================================================

CAT_COLS_FINAL = [
    "Driver",
    "Compound",
    "Race",
    "Year_str",
    "Driver_Compound",
    "Race_Compound",
    "Race_Year",
    "Driver_Race",
    "Driver_Year",
    "Compound_TyreLifeBin",
    "Compound_RaceProgressBin",
    "Stint_Compound",
]
CAT_COLS_FINAL = [c for c in CAT_COLS_FINAL if c in train_fe.columns and c in test_fe.columns]

# 020差分: add progress-window categorical columns
for c in CFG.PROGRESS_WINDOW_CAT_COLS:
    if c in train_fe.columns and c in test_fe.columns and c in orig_fe.columns:
        if c not in CAT_COLS_FINAL:
            CAT_COLS_FINAL.append(c)

# 025差分: add minimal Year/Stint categorical crosses as ordinal category, not TE.
for c in YEAR_STINT_CROSS_FEATURES_025:
    if c in train_fe.columns and c in test_fe.columns and c in orig_fe.columns:
        if c not in CAT_COLS_FINAL:
            CAT_COLS_FINAL.append(c)

print("020 CAT_COLS_FINAL added:", [c for c in CFG.PROGRESS_WINDOW_CAT_COLS if c in CAT_COLS_FINAL])
print("025 CAT_COLS_FINAL added:", [c for c in YEAR_STINT_CROSS_FEATURES_025 if c in CAT_COLS_FINAL])

DROP_FROM_X = [
    CFG.ID_COL,
    CFG.TARGET,
    "_dataset",
    "PitStop",  # shared_003 drops PitStop as post-fact label leakage
]

FEATURE_COLS_BASE = [
    c for c in train_fe.columns
    if c not in DROP_FROM_X and c in test_fe.columns
]

# Keep columns that also exist in original.
FEATURE_COLS_BASE = [
    c for c in FEATURE_COLS_BASE
    if c in orig_fe.columns
]

print("base feature count before TE:", len(FEATURE_COLS_BASE))
print("CAT_COLS_FINAL:", len(CAT_COLS_FINAL), CAT_COLS_FINAL)

# Ordinal encode selected categorical features.
oe = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=np.int32,
)

encode_base = pd.concat(
    [
        train_fe[CAT_COLS_FINAL],
        orig_fe[CAT_COLS_FINAL],
        test_fe[CAT_COLS_FINAL],
    ],
    axis=0,
    ignore_index=True,
).astype(str)

oe.fit(encode_base)

for df in [train_fe, orig_fe, test_fe]:
    df[CAT_COLS_FINAL] = oe.transform(df[CAT_COLS_FINAL].astype(str))

# Ensure all model features are numeric.
for df in [train_fe, orig_fe, test_fe]:
    for col in FEATURE_COLS_BASE:
        if df[col].dtype == "object" or str(df[col].dtype).startswith("category") or str(df[col].dtype).startswith("string"):
            codes, _ = pd.factorize(df[col].astype(str), sort=False)
            df[col] = codes.astype(np.int32)
        elif df[col].dtype == "float64":
            df[col] = df[col].astype(np.float32)

020 CAT_COLS_FINAL added: ['RaceProgressBin_20', 'Race_RaceProgressBin_20', 'Race_Compound_RaceProgressBin_20']
025 CAT_COLS_FINAL added: ['Year_Stint_025', 'Year_Compound_025']
base feature count before TE: 93
CAT_COLS_FINAL: 17 ['Driver', 'Compound', 'Race', 'Year_str', 'Driver_Compound', 'Race_Compound', 'Race_Year', 'Driver_Race', 'Driver_Year', 'Compound_TyreLifeBin', 'Compound_RaceProgressBin', 'Stint_Compound', 'RaceProgressBin_20', 'Race_RaceProgressBin_20', 'Race_Compound_RaceProgressBin_20', 'Year_Stint_025', 'Year_Compound_025']


In [11]:
# ============================================================
# Fold-safe Target Encoding
# ============================================================

TE_COLUMNS = [
    "Driver",
    "Race_Year",
    "Driver_Race",
    "Driver_Year",
    "Race",
    "Stint_Compound",
]
TE_COLUMNS = [c for c in TE_COLUMNS if c in train_fe.columns and c in test_fe.columns and c in orig_fe.columns]

# Explicitly do NOT add Year_Stint_025 / Year_Compound_025 to TE_COLUMNS at first pass.
if not CFG.ADD_YEAR_STINT_TE_025:
    TE_COLUMNS = [c for c in TE_COLUMNS if c not in YEAR_STINT_CROSS_FEATURES_025]

TE_NAMES = [f"TE_{c}" for c in TE_COLUMNS]

print("TE_COLUMNS:", TE_COLUMNS)


def add_fold_target_encoding(X_tr, y_tr, X_va, X_te, X_orig=None):
    X_tr = X_tr.copy()
    X_va = X_va.copy()
    X_te = X_te.copy()

    if X_orig is not None:
        X_orig = X_orig.copy()

    if not CFG.USE_TARGET_ENCODING or len(TE_COLUMNS) == 0:
        return X_tr, X_va, X_te, X_orig

    te = TargetEncoder(
        cv=5,
        smooth="auto",
        shuffle=True,
        random_state=CFG.SEED,
    )

    tr_encoded = te.fit_transform(X_tr[TE_COLUMNS], y_tr)
    va_encoded = te.transform(X_va[TE_COLUMNS])
    te_encoded = te.transform(X_te[TE_COLUMNS])

    X_tr[TE_NAMES] = tr_encoded
    X_va[TE_NAMES] = va_encoded
    X_te[TE_NAMES] = te_encoded

    if X_orig is not None:
        # Original rows are already included in X_tr in this implementation.
        # This branch is kept only for interface symmetry.
        pass

    return X_tr, X_va, X_te, X_orig

TE_COLUMNS: ['Driver', 'Race_Year', 'Driver_Race', 'Driver_Year', 'Race', 'Stint_Compound']


In [12]:
# ============================================================
# 020: Fold-safe progress-window count/frequency + TE
# ============================================================

def add_fold_count_frequency_features(
    X_tr: pd.DataFrame,
    X_va: pd.DataFrame,
    X_te: pd.DataFrame,
    cols: list,
):
    """
    Add fold-safe count/frequency features.
    Count maps are fitted on X_tr only.
    """
    X_tr = X_tr.copy()
    X_va = X_va.copy()
    X_te = X_te.copy()

    added = []
    denom = max(len(X_tr), 1)

    for col in cols:
        if col not in X_tr.columns:
            print(f"[020 count/freq skip] {col} not in X_tr")
            continue

        key_tr = X_tr[col].astype(str)
        key_va = X_va[col].astype(str)
        key_te = X_te[col].astype(str)

        vc = key_tr.value_counts(dropna=False)

        cnt_col = f"{col}_fold_count"
        frq_col = f"{col}_fold_freq"

        X_tr[cnt_col] = key_tr.map(vc).fillna(0).astype(np.float32)
        X_va[cnt_col] = key_va.map(vc).fillna(0).astype(np.float32)
        X_te[cnt_col] = key_te.map(vc).fillna(0).astype(np.float32)

        X_tr[frq_col] = (X_tr[cnt_col] / denom).astype(np.float32)
        X_va[frq_col] = (X_va[cnt_col] / denom).astype(np.float32)
        X_te[frq_col] = (X_te[cnt_col] / denom).astype(np.float32)

        added.extend([cnt_col, frq_col])

    return X_tr, X_va, X_te, added


def add_fold_progress_window_target_encoding(
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    X_va: pd.DataFrame,
    X_te: pd.DataFrame,
    cols: list,
    cv: int = 5,
    smooth="auto",
    seed: int = 3407,
):
    """
    Add fold-safe TargetEncoder features for progress-window categorical columns.
    Encoder is fit only on outer-fold X_tr/y_tr.
    """
    X_tr = X_tr.copy()
    X_va = X_va.copy()
    X_te = X_te.copy()

    use_cols = [c for c in cols if c in X_tr.columns]

    if len(use_cols) == 0:
        print("[020 TE skip] no progress-window TE columns found")
        return X_tr, X_va, X_te, []

    te = TargetEncoder(
        cv=cv,
        smooth=smooth,
        shuffle=True,
        random_state=seed,
    )

    out_names = [f"TE_{c}" for c in use_cols]

    X_tr[out_names] = te.fit_transform(X_tr[use_cols].astype(str), y_tr)
    X_va[out_names] = te.transform(X_va[use_cols].astype(str))
    X_te[out_names] = te.transform(X_te[use_cols].astype(str))

    for c in out_names:
        X_tr[c] = X_tr[c].astype(np.float32)
        X_va[c] = X_va[c].astype(np.float32)
        X_te[c] = X_te[c].astype(np.float32)

    return X_tr, X_va, X_te, out_names

In [13]:
# ============================================================
# CV setup
# ============================================================

print_section("CV Setup")

y_comp = train_fe[CFG.TARGET].astype(int).values
y_orig = orig_fe[CFG.TARGET].astype(int).values

strat_key = train_fe[CFG.TARGET].astype(str) + "__" + train_fe["Year"].astype(str)

folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(train_fe, strat_key)
)

for i, (_, va_idx) in enumerate(folds, 1):
    print(
        f"fold {i}: valid rows={len(va_idx)}, valid rate={train_fe.iloc[va_idx][CFG.TARGET].mean():.6f}"
    )


CV Setup
fold 1: valid rows=87828, valid rate=0.198991
fold 2: valid rows=87828, valid rate=0.198991
fold 3: valid rows=87828, valid rate=0.198968
fold 4: valid rows=87828, valid rate=0.198968
fold 5: valid rows=87828, valid rate=0.198991


In [14]:
# ============================================================
# LightGBM training
# ============================================================

def make_lgb_model(seed: int, device: str):
    return lgb.LGBMClassifier(
        n_estimators=CFG.LGB_N_ESTIMATORS,
        learning_rate=CFG.LGB_LEARNING_RATE,
        num_leaves=CFG.LGB_NUM_LEAVES,
        min_child_samples=CFG.LGB_MIN_CHILD_SAMPLES,
        subsample=CFG.LGB_SUBSAMPLE,
        subsample_freq=CFG.LGB_SUBSAMPLE_FREQ,
        colsample_bytree=CFG.LGB_COLSAMPLE_BYTREE,
        reg_alpha=CFG.LGB_REG_ALPHA,
        reg_lambda=CFG.LGB_REG_LAMBDA,
        max_bin=CFG.LGB_MAX_BIN,
        class_weight="balanced",
        device=device,
        gpu_use_dp=False,
        random_state=seed,
        n_jobs=-1,
        verbose=-1,
    )


def fit_lgb_with_fallback(model, X_tr, y_tr, X_va, y_va):
    try:
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[
                lgb.early_stopping(CFG.LGB_EARLY_STOPPING, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        return model, model.get_params().get("device", "unknown")
    except Exception as e:
        print("LightGBM GPU/device training failed. Retrying with CPU.")
        print("Error:", repr(e))

        cpu_model = make_lgb_model(
            seed=model.get_params().get("random_state", CFG.SEED),
            device="cpu",
        )
        cpu_model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[
                lgb.early_stopping(CFG.LGB_EARLY_STOPPING, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        return cpu_model, "cpu"


print_section("Training LightGBM full-FE single + Year/Stint flags")

oof = np.zeros(len(train_fe), dtype=np.float32)
pred = np.zeros(len(test_fe), dtype=np.float32)

fold_scores = []
fold_seed_scores = []
best_iterations = []
used_devices = []
feature_importance_list = []
progress_window_features_final = []

X_test_base = test_fe[FEATURE_COLS_BASE].copy()

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    print("\n" + "=" * 80)
    print(f"Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 80)

    X_tr_comp = train_fe.iloc[tr_idx][FEATURE_COLS_BASE].copy()
    y_tr_comp = train_fe.iloc[tr_idx][CFG.TARGET].astype(int).reset_index(drop=True)

    X_va = train_fe.iloc[va_idx][FEATURE_COLS_BASE].copy()
    y_va = train_fe.iloc[va_idx][CFG.TARGET].astype(int).values

    if CFG.USE_ORIGINAL_ROWS:
        X_orig_fold = orig_fe[FEATURE_COLS_BASE].copy()
        y_orig_fold = orig_fe[CFG.TARGET].astype(int).reset_index(drop=True)

        X_tr = pd.concat(
            [X_tr_comp.reset_index(drop=True), X_orig_fold.reset_index(drop=True)],
            axis=0,
            ignore_index=True,
        )
        y_tr = pd.concat(
            [y_tr_comp, y_orig_fold],
            axis=0,
            ignore_index=True,
        )
    else:
        X_tr = X_tr_comp.reset_index(drop=True)
        y_tr = y_tr_comp

    X_te = X_test_base.copy()

    X_tr, X_va, X_te, _ = add_fold_target_encoding(
        X_tr=X_tr,
        y_tr=y_tr,
        X_va=X_va,
        X_te=X_te,
        X_orig=None,
    )

    # ============================================================
    # 020差分: progress-window count/frequency + TE
    # ============================================================

    if CFG.USE_PROGRESS_WINDOW_TE:
        X_tr, X_va, X_te, progress_count_features = add_fold_count_frequency_features(
            X_tr=X_tr,
            X_va=X_va,
            X_te=X_te,
            cols=CFG.PROGRESS_WINDOW_CAT_COLS,
        )

        X_tr, X_va, X_te, progress_te_features = add_fold_progress_window_target_encoding(
            X_tr=X_tr,
            y_tr=y_tr,
            X_va=X_va,
            X_te=X_te,
            cols=CFG.PROGRESS_WINDOW_TE_COLS,
            cv=5,
            smooth="auto",
            seed=CFG.SEED,
        )

        if fold == 1:
            progress_window_features_final = progress_count_features + progress_te_features
            print("020 progress count/freq features:", progress_count_features)
            print("020 progress TE features:", progress_te_features)
            print("020 progress total added:", len(progress_window_features_final))

        # Guardrails
        assert "Race_Year_RaceProgressBin_20" not in X_tr.columns
        assert "TE_Race_Year_RaceProgressBin_20" not in X_tr.columns
        assert "Race_RaceProgressBin_20" in X_tr.columns
        assert "Race_Compound_RaceProgressBin_20" in X_tr.columns
        assert "TE_Race_RaceProgressBin_20" in X_tr.columns
        assert "TE_Race_Compound_RaceProgressBin_20" in X_tr.columns

    if fold == 1:
        final_features = X_tr.columns.tolist()
        print("final feature count:", len(final_features))
        print("025 features in final:", [c for c in final_features if c.endswith("_025")])
        print("final features:", final_features)

    print("X_tr:", X_tr.shape, "target mean:", float(np.mean(y_tr)))
    print("X_va:", X_va.shape, "target mean:", float(np.mean(y_va)))
    print("X_te:", X_te.shape)

    fold_val = np.zeros(len(va_idx), dtype=np.float32)
    fold_test = np.zeros(len(test_fe), dtype=np.float32)

    for seed in CFG.SEEDS_BASE:
        print(f"  seed {seed}")

        model = make_lgb_model(seed=seed, device=CFG.LGB_DEVICE)
        model, used_device = fit_lgb_with_fallback(
            model=model,
            X_tr=X_tr,
            y_tr=y_tr,
            X_va=X_va,
            y_va=y_va,
        )

        p_va = model.predict_proba(X_va)[:, 1].astype(np.float32)
        p_te = model.predict_proba(X_te)[:, 1].astype(np.float32)

        fold_val += p_va / len(CFG.SEEDS_BASE)
        fold_test += p_te / len(CFG.SEEDS_BASE)

        seed_auc = roc_auc_score(y_va, p_va)
        fold_seed_scores.append(
            {
                "fold": fold,
                "seed": seed,
                "auc": float(seed_auc),
                "best_iteration": int(getattr(model, "best_iteration_", -1) or -1),
                "used_device": used_device,
            }
        )

        best_iterations.append(int(getattr(model, "best_iteration_", -1) or -1))
        used_devices.append(used_device)

        fi = pd.DataFrame(
            {
                "feature": X_tr.columns.tolist(),
                "importance": model.feature_importances_,
                "fold": fold,
                "seed": seed,
            }
        )
        feature_importance_list.append(fi)

        del model
        gc.collect()

    oof[va_idx] = fold_val
    pred += fold_test / CFG.N_SPLITS

    fold_auc = roc_auc_score(y_va, fold_val)
    fold_scores.append(float(fold_auc))

    print(f"Fold {fold} AUC: {fold_auc:.8f}")

    del X_tr, X_va, X_te
    gc.collect()


cv_auc = roc_auc_score(y_comp, oof)

print_section("CV Result")
print("CV AUC:", cv_auc)
print("fold_scores:", fold_scores)
print("fold_mean:", float(np.mean(fold_scores)))
print("fold_std :", float(np.std(fold_scores)))
print("best_iterations:", best_iterations)
print("used_devices:", sorted(set(used_devices)))


Training LightGBM full-FE single + Year/Stint flags

Fold 1/5
020 progress count/freq features: ['RaceProgressBin_20_fold_count', 'RaceProgressBin_20_fold_freq', 'Race_RaceProgressBin_20_fold_count', 'Race_RaceProgressBin_20_fold_freq', 'Race_Compound_RaceProgressBin_20_fold_count', 'Race_Compound_RaceProgressBin_20_fold_freq']
020 progress TE features: ['TE_Race_RaceProgressBin_20', 'TE_Race_Compound_RaceProgressBin_20']
020 progress total added: 8
final feature count: 107
025 features in final: ['Is_2022_025', 'Is_2023_025', 'Is_2024_025', 'Is_2025_025', 'Is_Stint1_025', 'Is_Stint2_025', 'Is_Stint3_025', 'Is_Stint4Plus_025', 'Is_Second_Stint_025', 'Is_Third_StintPlus_025', 'TyreLife_sq_025', 'TyreLife_sqrt_025', 'TyreLife_log1p_025', 'LapTime_Delta_abs_025', 'Cumulative_Degradation_abs_025', 'Stint_x_RaceProgress_025', 'Stint_x_TyreLife_sq_025', 'TyreLife_sq_x_RaceProgress_025', 'Year_Stint_025', 'Year_Compound_025']
final features: ['Driver', 'Compound', 'Race', 'Year', 'LapNumber'

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


  seed 42
Fold 1 AUC: 0.95036914

Fold 2/5
X_tr: (452683, 107) target mean: 0.21147911452385001
X_va: (87828, 107) target mean: 0.19899121009245344
X_te: (188165, 107)
  seed 3407
  seed 42
Fold 2 AUC: 0.95132607

Fold 3/5
X_tr: (452683, 107) target mean: 0.21148353262658418
X_va: (87828, 107) target mean: 0.1989684383112447
X_te: (188165, 107)
  seed 3407
  seed 42
Fold 3 AUC: 0.95081252

Fold 4/5
X_tr: (452683, 107) target mean: 0.21148353262658418
X_va: (87828, 107) target mean: 0.1989684383112447
X_te: (188165, 107)
  seed 3407
  seed 42
Fold 4 AUC: 0.95127217

Fold 5/5
X_tr: (452683, 107) target mean: 0.21147911452385001
X_va: (87828, 107) target mean: 0.19899121009245344
X_te: (188165, 107)
  seed 3407
  seed 42
Fold 5 AUC: 0.95186470

CV Result
CV AUC: 0.9511230993013209
fold_scores: [0.9503691373184895, 0.9513260690997236, 0.9508125236506595, 0.9512721689378926, 0.9518647025627576]
fold_mean: 0.9511289203139045
fold_std : 0.0005055967520437788
best_iterations: [9000, 9000, 9000

In [15]:
# ============================================================
# Save artifacts
# ============================================================

print_section("Save Artifacts")

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", pred)

pd.DataFrame(
    {
        CFG.ID_COL: train_ids,
        CFG.TARGET: oof,
    }
).to_csv(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv", index=False)

sub_out = sub.copy()
target_col = [c for c in sub_out.columns if c != CFG.ID_COL][0]
sub_out[target_col] = pred

sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub_out.to_csv(sub_path, index=False)

final_features = feature_importance_list[0]["feature"].tolist()

feature_df = pd.DataFrame(
    {
        "feature": final_features,
        "is_cat_encoded": [c in CAT_COLS_FINAL for c in final_features],
        "is_te": [c in TE_NAMES for c in final_features],
        "is_025_year_stint": [c.endswith("_025") for c in final_features],
        "is_025_flag": [c in YEAR_STINT_FLAG_FEATURES_025 for c in final_features],
        "is_025_cross": [c in YEAR_STINT_CROSS_FEATURES_025 for c in final_features],
        "is_progress_window_base": [c in CFG.PROGRESS_WINDOW_CAT_COLS for c in final_features],
        "is_progress_window_count_freq": [
            c in progress_window_features_final and (c.endswith("_fold_count") or c.endswith("_fold_freq"))
            for c in final_features
        ],
        "is_progress_window_te": [
            c in progress_window_features_final and c.startswith("TE_")
            for c in final_features
        ],
        "is_progress_window_any": [
            (
                c in CFG.PROGRESS_WINDOW_CAT_COLS
                or c in progress_window_features_final
            )
            for c in final_features
        ],
        "is_freq": [c.endswith("_freq") for c in final_features],
        "is_lag": [
            c in [
                "Delta_lag1",
                "Delta_roll3_mean",
                "Deg_diff",
                "TyreLife_growth",
                "LapTime_lag1",
                "LapTime_diff",
            ]
            for c in final_features
        ],
        "contains_race": ["Race" in c for c in final_features],
        "contains_compound": ["Compound" in c for c in final_features],
        "contains_progress": ["RaceProgress" in c for c in final_features],
        "contains_year": ["Year" in c for c in final_features],
        "contains_stint": ["Stint" in c for c in final_features],
    }
)
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

pd.DataFrame(
    {
        "feature": progress_window_features_final,
    }
).to_csv(CFG.OUTDIR / "progress_window_features_020.csv", index=False)

pd.DataFrame(
    {
        "base_progress_window_feature": CFG.PROGRESS_WINDOW_CAT_COLS,
    }
).to_csv(CFG.OUTDIR / "progress_window_base_features_020.csv", index=False)

pd.DataFrame(
    {
        "feature": [c for c in final_features if c.endswith("_025")],
    }
).to_csv(CFG.OUTDIR / "year_stint_features_025.csv", index=False)

# Guardrails for saved features
assert "Race_RaceProgressBin_20" in final_features
assert "Race_Compound_RaceProgressBin_20" in final_features
assert "TE_Race_RaceProgressBin_20" in final_features
assert "TE_Race_Compound_RaceProgressBin_20" in final_features
assert "Race_Year_RaceProgressBin_20" not in final_features
assert "TE_Race_Year_RaceProgressBin_20" not in final_features
assert CFG.DANGER_COL not in final_features
assert "Stint_x_Normalized" not in final_features
assert "Norm_x_Hardness" not in final_features

# Avoid accidental TE for added Year/Stint crosses at first pass.
assert "TE_Year_Stint_025" not in final_features
assert "TE_Year_Compound_025" not in final_features

fi_all = pd.concat(feature_importance_list, axis=0, ignore_index=True)
fi_summary = (
    fi_all
    .groupby("feature", as_index=False)["importance"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("mean", ascending=False)
)

fi_all.to_csv(CFG.OUTDIR / "feature_importance_by_fold_seed.csv", index=False)
fi_summary.to_csv(CFG.OUTDIR / "feature_importance.csv", index=False)

fold_seed_df = pd.DataFrame(fold_seed_scores)
fold_seed_df.to_csv(CFG.OUTDIR / "fold_seed_scores.csv", index=False)

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-13",
    },
    "base": {
        "experiment": "exp_20260511_020_lgb_progress_window_te_min",
        "code_policy": "020 implementation preserved; only Year/Stint explicit flags added.",
    },
    "source": {
        "shared_code": "shared_003",
        "extracted_component": "LightGBM full-FE single",
        "note": [
            "Not a full shared_003 reproduction.",
            "Only LGB full-FE member is extracted.",
            "Competition OOF is created with competition validation rows only.",
        ],
    },
    "data": {
        "train_shape": list(train_raw.shape),
        "test_shape": list(test_raw.shape),
        "original_shape_after_drop": list(orig_raw.shape),
        "target": CFG.TARGET,
        "id_col": CFG.ID_COL,
        "danger_col": CFG.DANGER_COL,
        "danger_col_used": False,
        "use_original_rows": CFG.USE_ORIGINAL_ROWS,
        "competition_target_mean": float(train_raw[CFG.TARGET].mean()),
        "original_target_mean": float(orig_raw[CFG.TARGET].mean()),
    },
    "features": {
        "base_feature_count_before_te": len(FEATURE_COLS_BASE),
        "final_feature_count": len(final_features),
        "cat_cols_final": CAT_COLS_FINAL,
        "te_columns": TE_COLUMNS,
        "te_names": TE_NAMES,
        "drop_from_x": DROP_FROM_X,
        "year_stint_025": {
            "enabled": True,
            "flag_features": [c for c in YEAR_STINT_FLAG_FEATURES_025 if c in final_features],
            "cross_features": [c for c in YEAR_STINT_CROSS_FEATURES_025 if c in final_features],
            "cross_frequency_features": [
                c for c in final_features
                if c in ["Year_Stint_025_freq", "Year_Compound_025_freq"]
            ],
            "te_enabled_for_added_crosses": CFG.ADD_YEAR_STINT_TE_025,
            "not_added": [
                "Normalized_TyreLife",
                "Stint_x_Normalized",
                "Norm_x_Hardness",
                "TyreLife / inferred_stint_length",
                "final_stint_length_proxy",
                "future_lap_or_endpoint_proxy",
                "TE_Year_Stint_025",
                "TE_Year_Compound_025",
            ],
        },
        "progress_window_020": {
            "enabled": True,
            "bins": CFG.PROGRESS_BINS,
            "base_features": CFG.PROGRESS_WINDOW_CAT_COLS,
            "te_source_features": CFG.PROGRESS_WINDOW_TE_COLS,
            "added_features": progress_window_features_final,
            "count_frequency_features": [
                c for c in progress_window_features_final
                if c.endswith("_fold_count") or c.endswith("_fold_freq")
            ],
            "target_encoding_features": [
                c for c in progress_window_features_final
                if c.startswith("TE_")
            ],
            "not_added": [
                "Race_Year_RaceProgressBin_20",
                "TE_Race_Year_RaceProgressBin_20",
                "72bin progress features",
                "pseudo label",
                "original prior",
            ],
            "rationale": [
                "EDA shows Race-specific pit windows along RaceProgress.",
                "Race x RaceProgressBin directly represents race-specific pit windows.",
                "Race x Compound x RaceProgressBin adds tyre strategy context.",
                "Year is intentionally excluded from progress-window interaction to avoid amplifying Year artifact.",
            ],
        },
        "notes": [
            "shared_003-style full FE.",
            "025 adds explicit Year/Stint/Tyre-wear signal features.",
            "020 adds progress-window features from Race x RaceProgressBin.",
            "PitStop is dropped following shared_003.",
            "Target x Year stratification is used.",
            "Original rows are appended to fold train side only.",
            "OOF target encoding is created inside each fold.",
            "020 progress-window TE is also created inside each fold.",
            "025 added Year/Stint crosses are not target-encoded in this first pass.",
        ],
    },
    "model": {
        "family": "LightGBM",
        "estimator": "LGBMClassifier",
        "seeds": CFG.SEEDS_BASE,
        "params": {
            "n_estimators": CFG.LGB_N_ESTIMATORS,
            "learning_rate": CFG.LGB_LEARNING_RATE,
            "num_leaves": CFG.LGB_NUM_LEAVES,
            "min_child_samples": CFG.LGB_MIN_CHILD_SAMPLES,
            "subsample": CFG.LGB_SUBSAMPLE,
            "subsample_freq": CFG.LGB_SUBSAMPLE_FREQ,
            "colsample_bytree": CFG.LGB_COLSAMPLE_BYTREE,
            "reg_alpha": CFG.LGB_REG_ALPHA,
            "reg_lambda": CFG.LGB_REG_LAMBDA,
            "max_bin": CFG.LGB_MAX_BIN,
            "class_weight": "balanced",
            "early_stopping_rounds": CFG.LGB_EARLY_STOPPING,
            "device_requested": CFG.LGB_DEVICE,
            "devices_used": sorted(set(used_devices)),
        },
        "cv": {
            "scheme": "StratifiedKFold",
            "stratification": "target x Year",
            "n_splits": CFG.N_SPLITS,
            "shuffle": True,
            "random_state": CFG.SEED,
        },
    },
    "result": {
        "cv_auc": float(cv_auc),
        "fold_scores": fold_scores,
        "fold_mean": float(np.mean(fold_scores)),
        "fold_std": float(np.std(fold_scores)),
        "fold_seed_scores": fold_seed_scores,
        "best_iterations": best_iterations,
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof_npy": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred_npy": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "oof_csv": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv"),
        "submission": str(sub_path),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "feature_importance": str(CFG.OUTDIR / "feature_importance.csv"),
        "cv_result": str(CFG.OUTDIR / "cv_result.json"),
        "progress_window_features": str(CFG.OUTDIR / "progress_window_features_020.csv"),
        "progress_window_base_features": str(CFG.OUTDIR / "progress_window_base_features_020.csv"),
        "year_stint_features": str(CFG.OUTDIR / "year_stint_features_025.csv"),
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)


Save Artifacts


In [16]:
# memo draft
memo_yml = f"""experiment:
  id: exp_20260513_025_lgb_year_stint_flags_min
  title: LGB progress-window TE + Year/Stint explicit flags
  competition: PS S6E5 Predicting F1 Pit Stops
  metric: AUC
  status: completed

objective:
  summary: >
    020 LGB progress-window TE を完全なbaseとして維持し、
    discussion由来の Is_2023 / Stint flags / tyre-wear signal を明示特徴として追加する。
    Year artifact と Stint 周辺の明示化が020 LGB branchの単体CV/LBまたはblend寄与を改善するか確認する。
  base_experiment: exp_20260511_020_lgb_progress_window_te_min
  intended_role: 020_replacement_candidate

implementation_check:
  base_code: code_020.txt
  preserved_from_020:
    - shared_003_style_full_FE
    - original_rows_appended_to_each_fold_train
    - target_x_year_stratification
    - seed_ensemble_3407_42
    - lgb_params_from_020
    - gpu_with_cpu_fallback
    - progress_window_base_features
    - progress_window_count_frequency
    - progress_window_target_encoding
  changed_from_020:
    - add_year_stint_flags_025
    - add_minimal_year_stint_crosses_025
    - add_frequency_for_year_stint_crosses
  not_added:
    - weather_features
    - Race_Year_RaceProgressBin
    - 72_bins
    - original_prior
    - pseudo_label
    - Normalized_TyreLife_or_proxy
    - Stint_x_Normalized
    - Norm_x_Hardness
    - TyreLife_div_inferred_stint_length
    - future_or_endpoint_proxy
    - TE_Year_Stint_025
    - TE_Year_Compound_025

data:
  competition_train: /kaggle/input/competitions/playground-series-s6e5/train.csv
  competition_test: /kaggle/input/competitions/playground-series-s6e5/test.csv
  original: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv
  target: PitNextLap
  danger_col:
    name: Normalized_TyreLife
    used: false

features:
  inherited_from_020:
    - shared_003_full_FE
    - RaceProgressBin_20
    - Race_RaceProgressBin_20
    - Race_Compound_RaceProgressBin_20
    - fold_safe_count_frequency_for_progress_window
    - fold_safe_TE_for_progress_window
  added_025:
    year_flags:
      - Is_2022_025
      - Is_2023_025
      - Is_2024_025
      - Is_2025_025
    stint_flags:
      - Is_Stint1_025
      - Is_Stint2_025
      - Is_Stint3_025
      - Is_Stint4Plus_025
      - Is_Second_Stint_025
      - Is_Third_StintPlus_025
    tyre_wear:
      - TyreLife_sq_025
      - TyreLife_sqrt_025
      - TyreLife_log1p_025
      - LapTime_Delta_abs_025
      - Cumulative_Degradation_abs_025
    interactions:
      - Stint_x_RaceProgress_025
      - Stint_x_TyreLife_sq_025
      - TyreLife_sq_x_RaceProgress_025
    categorical_cross:
      - Year_Stint_025
      - Year_Compound_025
    frequency:
      - Year_Stint_025_freq
      - Year_Compound_025_freq
  te_policy:
    added_year_stint_te: false
    reason: >
      初手からYear/Stint系にTEをかけるとYear artifactを過剰増幅しやすいため、
      025ではraw/ordinal/frequencyまでに留める。

model:
  family: LightGBM
  seeds:
    - 3407
    - 42
  cv:
    scheme: StratifiedKFold
    stratification: target_x_Year
    n_splits: 5
    shuffle: true
    random_state: 3407
  params:
    n_estimators: 9000
    learning_rate: 0.022
    num_leaves: 31
    min_child_samples: 120
    subsample: 0.88
    subsample_freq: 1
    colsample_bytree: 0.82
    reg_alpha: 0.25
    reg_lambda: 10.0
    max_bin: 255
    class_weight: balanced
    early_stopping_rounds: 450

result:
  cv_auc: {cv_auc}
  public_lb: null
  fold_scores: {fold_scores}
  compared_to_020:
    cv_020: 0.9510267120359983
    public_lb_020: 0.95075
    delta_cv: {cv_auc-0.9510267120359983}
    delta_lb: null
  compared_to_024:
    cv_024: 0.9511783121413456
    public_lb_024: 0.95086
    delta_cv: {cv_auc-0.9511783121413456}
    delta_lb: null

artifacts:
  notebook: exp_20260513_025_lgb_year_stint_flags_min.ipynb
  oof: oof_exp_20260513_025_lgb_year_stint_flags_min.npy
  pred: pred_exp_20260513_025_lgb_year_stint_flags_min.npy
  submission: submission_exp_20260513_025_lgb_year_stint_flags_min.csv
  feature_importance: feature_importance.csv
  feature_columns: feature_columns.csv
  year_stint_features: year_stint_features_025.csv

blend_policy:
  benchmark_code: code_010_add_slsqp.txt
  compare_methods:
    - avg
    - ridge_meta_all
    - logreg_meta_all
    - hc_nonnegative_raw
    - nm_softmax_raw
    - slsqp_signed_raw_neg005
  add_or_replace: >
    020/024置換候補として評価する。
    020/024よりCV/LBが改善する、またはCV同等でも既存stackとの相関が下がる場合のみ
    blend benchmarkに投入する。

blend_with_025:
  source_notebook: ps-s6e5-010-blend-core-nm-neg-weight.ipynb
  source_json: blend_result.json
  benchmark_code: code_010_add_slsqp.txt
  stack:
    - "007"
    - "014"
    - "015"
    - "016"
    - "017"
    - "018"
    - "020"
    - "021"
    - "022"
    - "023"
    - "025"
  results:
    avg:
      cv_auc: null
      public_lb: null
      weights:
        "007": null
        "014": null
        "015": null
        "016": null
        "017": null
        "018": null
        "020": null
        "021": null
        "022": null
        "023": null
        "025": null
    ridge_meta_all:
      cv_auc: null
      public_lb: null
      params:
        alpha: null
    logreg_meta_all:
      cv_auc: null
      public_lb: null
      params:
        C: null
    hc_nonnegative_raw:
      cv_auc: null
      public_lb: null
      weights:
        "007": null
        "014": null
        "015": null
        "016": null
        "017": null
        "018": null
        "020": null
        "021": null
        "022": null
        "023": null
        "025": null
    nm_softmax_raw:
      cv_auc: null
      public_lb: null
      weights:
        "007": null
        "014": null
        "015": null
        "016": null
        "017": null
        "018": null
        "020": null
        "021": null
        "022": null
        "023": null
        "025": null
    slsqp_signed_raw_neg005:
      cv_auc: null
      public_lb: null
      weights:
        "007": null
        "014": null
        "015": null
        "016": null
        "017": null
        "018": null
        "020": null
        "021": null
        "022": null
        "023": null
        "025": null
      params:
        neg_bound: -0.05
        pos_bound: 0.5
      diagnostics:
        min_weight: null
        max_weight: null
        negative_weight_models: null

judgement:
  status: pending
  summary: >
    CV/LB/相関確認後に keep / hold / drop を判断する。
"""
with open(CFG.OUTDIR / "memo_draft.yml", "w", encoding="utf-8") as f:
    f.write(memo_yml)

print("Saved to:", CFG.OUTDIR)
print("Submission:", sub_path)

display(fi_summary.head(80))
display(sub_out.head())

Saved to: /kaggle/working/exp_20260513_025_lgb_year_stint_flags_min
Submission: /kaggle/working/exp_20260513_025_lgb_year_stint_flags_min/submission_exp_20260513_025_lgb_year_stint_flags_min.csv


,index,feature,mean,std
79,79,TE_Driver_Race,9166.5,123.685309
22,22,EstimatedTotalLaps,8257.8,111.216306
40,40,LapTime (s),7865.7,105.406146
80,80,TE_Driver_Year,7815.9,113.847315
18,18,Driver_Race_freq,7531.0,131.023323
...,...,...,...,...
95,95,TyreLife_sqrt_025,318.5,18.246765
5,5,Compound_freq,316.8,18.629725
93,93,TyreLife_sq_025,310.7,20.736709
92,92,TyreLife_log1p_025,297.3,16.640312


,id,PitNextLap
0,439140,0.009142
1,439141,0.029197
2,439142,0.007526
3,439143,0.319134
4,439144,0.947553
